<a href="https://colab.research.google.com/github/madhavkhurana1005/AI-Projects/blob/main/imdb_project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# EXPERIMENTAL PROJECT
## 🎯 Does domain-adaptive MLM fine-tuning improve sentiment classification performance?




In [2]:
!pip install datasets evaluate transformers[sentencepiece]
!pip install accelerate
!apt install git-lfs

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 4.7 MB/s eta 0:00:00
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
git-lfs is already the newest version (3.0.2-1ubuntu0.3).
0 upgraded, 0 newly installed, 0 to remove and 37 not upgraded.


In [1]:
import torch
torch.cuda.is_available()

True

In [3]:
!git config --global user.email "madhavkhurana1005@gmail.com"
!git config --global user.name "Madhav Khurana"

In [4]:
from huggingface_hub import notebook_login

notebook_login()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [6]:
from datasets import load_dataset

imdb_data = load_dataset("imdb")
imdb_data

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 25000
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 25000
    })
    unsupervised: Dataset({
        features: ['text', 'label'],
        num_rows: 50000
    })
})

In [40]:
from transformers import AutoModelForMaskedLM
from transformers import AutoTokenizer

model_checkpoint = "distilbert-base-uncased"
model = AutoModelForMaskedLM.from_pretrained(model_checkpoint)

tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)

def tokenize_function(examples):
    return tokenizer(examples["text"])


def group_texts(examples):
    # concatenate all input_ids
    concatenated = {k: sum(examples[k], []) for k in examples.keys()}

    total_length = len(concatenated["input_ids"])

    # drop remainder
    total_length = (total_length // 256) * 256

    result = {
        k: [t[i:i+256] for i in range(0, total_length, 256)]
        for k, t in concatenated.items()
    }

    return result

from transformers import DataCollatorForLanguageModeling

data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm_probability=0.15)


Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

In [8]:
tokenized = imdb_data.map(tokenize_function, batched=True, remove_columns=["text", "label"])

lm_dataset = tokenized.map(group_texts, batched=True)
lm_dataset

Map:   0%|          | 0/25000 [00:00<?, ? examples/s]

Token indices sequence length is longer than the specified maximum sequence length for this model (720 > 512). Running this sequence through the model will result in indexing errors


Map:   0%|          | 0/25000 [00:00<?, ? examples/s]

Map:   0%|          | 0/50000 [00:00<?, ? examples/s]

Map:   0%|          | 0/25000 [00:00<?, ? examples/s]

Map:   0%|          | 0/25000 [00:00<?, ? examples/s]

Map:   0%|          | 0/50000 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 30639
    })
    test: Dataset({
        features: ['input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 29946
    })
    unsupervised: Dataset({
        features: ['input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 61465
    })
})

In [39]:
from transformers import TrainingArguments

downsampled_dataset = lm_dataset["train"]


batch_size = 64
logging_steps = 500
model_name = model_checkpoint

training_args = TrainingArguments(
    output_dir=f"{model_name}-finetuned-imdb",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    weight_decay=0.01,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=16,
    gradient_accumulation_steps=2,
    num_train_epochs=2,
    fp16=True,
    logging_steps=logging_steps,
    logging_dir="./logs",
    push_to_hub=True
)

`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


In [41]:
from transformers import Trainer

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=downsampled_dataset,
    eval_dataset=lm_dataset["test"],
    data_collator=data_collator
)

In [42]:
trainer.train()
trainer.save_model("mlm-distilbert-imdb-full")

Epoch,Training Loss,Validation Loss
1,No log,2.304249
2,4.973832,2.275977


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...ed-imdb/training_args.bin: 100%|##########| 5.20kB / 5.20kB            

  ...ed-imdb/model.safetensors:  60%|#####9    |  160MB /  268MB            

In [45]:
import math
math.exp(2.27) # Perplexity score

9.67940081407284

In [48]:
# testing outpt using pipeline after transfer learning

from transformers import pipeline

fill_mask = pipeline("fill-mask", model=model, tokenizer=tokenizer)

fill_mask("It was full of [MASK].")

[{'score': 0.08086500316858292,
  'token': 20096,
  'token_str': 'surprises',
  'sequence': 'it was full of surprises.'},
 {'score': 0.03357812762260437,
  'token': 2111,
  'token_str': 'people',
  'sequence': 'it was full of people.'},
 {'score': 0.020849134773015976,
  'token': 2668,
  'token_str': 'blood',
  'sequence': 'it was full of blood.'},
 {'score': 0.014612007886171341,
  'token': 7800,
  'token_str': 'secrets',
  'sequence': 'it was full of secrets.'},
 {'score': 0.013461211696267128,
  'token': 2293,
  'token_str': 'love',
  'sequence': 'it was full of love.'}]

# Classification on imdb data finetuned Masked Language Model

In [49]:
from transformers import AutoModelForSequenceClassification

model_cls_mlm = AutoModelForSequenceClassification.from_pretrained(
    "mlm-distilbert-imdb",
    num_labels=2
)

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: mlm-distilbert-imdb
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.weight | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
classifier.weight       | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.weight   | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [50]:
def tokenize_function_classification(examples):
    return tokenizer(
        examples["text"],
        truncation=True,
        padding="max_length",
        max_length=256
    )

In [51]:
tokenized_cls_mlm = imdb_data.map(
    tokenize_function_classification,
    batched=True,
    remove_columns=["text"]
)

Map:   0%|          | 0/25000 [00:00<?, ? examples/s]

Map:   0%|          | 0/25000 [00:00<?, ? examples/s]

Map:   0%|          | 0/50000 [00:00<?, ? examples/s]

In [52]:
training_args = TrainingArguments(
    output_dir=f"{model_name}-classification-imdb",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=16,
    num_train_epochs=2,
    weight_decay=0.01,
    fp16=True,
    push_to_hub=True
)

In [53]:
from sklearn.metrics import accuracy_score, f1_score

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = logits.argmax(axis=1)
    return {
        "accuracy": accuracy_score(labels, preds),
        "f1": f1_score(labels, preds),
    }

In [54]:
trainer_cls_mlm = Trainer(
    model=model_cls_mlm,
    args=training_args,
    train_dataset=tokenized_cls_mlm["train"],
    eval_dataset=tokenized_cls_mlm["test"],
    compute_metrics=compute_metrics,
)

In [55]:
trainer_cls_mlm.train()

Epoch,Training Loss,Validation Loss,Accuracy,F1
1,0.308914,0.225707,0.909400,0.908377
2,0.169714,0.237771,0.912760,0.913222


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=1564, training_loss=0.231149021012094, metrics={'train_runtime': 442.4795, 'train_samples_per_second': 113.0, 'train_steps_per_second': 3.535, 'total_flos': 3311684966400000.0, 'train_loss': 0.231149021012094, 'epoch': 2.0})

In [56]:
result_mlm = trainer_cls_mlm.evaluate()

In [57]:
result_mlm

{'eval_loss': 0.237771138548851,
 'eval_accuracy': 0.91276,
 'eval_f1': 0.9132216607647317,
 'eval_runtime': 51.0507,
 'eval_samples_per_second': 489.709,
 'eval_steps_per_second': 30.617,
 'epoch': 2.0}

# Classification on baseline Masked Language Model - Distilbert

In [32]:
from transformers import AutoModelForSequenceClassification

model_cls_baseline = AutoModelForSequenceClassification.from_pretrained(
    "distilbert-base-uncased",
    num_labels=2
)

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.weight | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
classifier.weight       | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.weight   | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [33]:
tokenized_cls_baseline = imdb_data.map(
    tokenize_function_classification,
    batched=True,
    remove_columns=["text"]
)

Map:   0%|          | 0/25000 [00:00<?, ? examples/s]

In [35]:
# tokenized_cls_baseline["train"][0]

In [36]:
trainer_cls_baseline = Trainer(
    model=model_cls_baseline,
    args=training_args,
    train_dataset=tokenized_cls_baseline["train"],
    eval_dataset=tokenized_cls_baseline["test"],
    compute_metrics=compute_metrics,
)

In [37]:
trainer_cls_baseline.train()

Epoch,Training Loss,Validation Loss,Accuracy,F1
1,0.317128,0.224042,0.909200,0.908549
2,0.172412,0.236022,0.912680,0.912816


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=1564, training_loss=0.23421350952304537, metrics={'train_runtime': 441.4322, 'train_samples_per_second': 113.268, 'train_steps_per_second': 3.543, 'total_flos': 3311684966400000.0, 'train_loss': 0.23421350952304537, 'epoch': 2.0})

In [38]:
result_baseline = trainer_cls_baseline.evaluate()
result_baseline

{'eval_loss': 0.23602186143398285,
 'eval_accuracy': 0.91268,
 'eval_f1': 0.9128160070290348,
 'eval_runtime': 50.8012,
 'eval_samples_per_second': 492.115,
 'eval_steps_per_second': 30.767,
 'epoch': 2.0}

# Error Analysis

In [58]:
import numpy as np

predictions = trainer_cls_mlm.predict(tokenized_cls_mlm["test"])

logits = predictions.predictions
labels = predictions.label_ids

preds = np.argmax(logits, axis=1)

In [59]:
wrong_indices = np.where(preds != labels)[0]

In [63]:
def label_to_text(label):
    return "positive" if label == 1 else "negative"


for i in wrong_indices[:10]:
    i = int(i)
    print("TEXT:", imdb_data["test"][i]["text"])
    print("ACTUAL:", label_to_text(labels[i]))
    print("PRED:", label_to_text(preds[i]))
    print("="*80)

TEXT: First off let me say, If you haven't enjoyed a Van Damme movie since bloodsport, you probably will not like this movie. Most of these movies may not have the best plots or best actors but I enjoy these kinds of movies for what they are. This movie is much better than any of the movies the other action guys (Segal and Dolph) have thought about putting out the past few years. Van Damme is good in the movie, the movie is only worth watching to Van Damme fans. It is not as good as Wake of Death (which i highly recommend to anyone of likes Van Damme) or In hell but, in my opinion it's worth watching. It has the same type of feel to it as Nowhere to Run. Good fun stuff!
ACTUAL: negative
PRED: positive
TEXT: Isaac Florentine has made some of the best western Martial Arts action movies ever produced. In particular US Seals 2, Cold Harvest, Special Forces and Undisputed 2 are all action classics. You can tell Isaac has a real passion for the genre and his films are always eventful, creati

## < APPENDIX >

In [13]:




# DataCollatorForLanguageModeling

# Key configs:

# mlm_probability = 0.15
# learning_rate = 2e-5
# epochs = 1–2

# Save model:

# trainer.save_model("mlm-distilbert-imdb")

In [29]:
for i in range(10):
  print(len(lm_dataset["train"]["input_ids"][i]))

# len(lm_dataset["train"]["input_ids"])
# lm_dataset

256
256
256
256
256
256
256
256
256
256


In [31]:
for i in range(10):
  print(len(tokenized["train"]["input_ids"][i]))

363
304
133
185
495
154
143
388
720
297


In [15]:
imdb_data["train"]

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 25000
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 25000
    })
    unsupervised: Dataset({
        features: ['text', 'label'],
        num_rows: 50000
    })
})

In [34]:
model_checkpoint.split("/")[-1]

'distilbert-base-uncased'

In [39]:
import transformers
print(transformers.__version__)

5.0.0


In [8]:
import torch

text = "This is a great [MASK]."

inputs = tokenizer(text, return_tensors="pt")
token_logits = model(**inputs).logits
# Find the location of [MASK] and extract its logits
mask_token_index = torch.where(inputs["input_ids"] == tokenizer.mask_token_id)[1]
mask_token_logits = token_logits[0, mask_token_index, :]
# Pick the [MASK] candidates with the highest logits
top_5_tokens = torch.topk(mask_token_logits, 5, dim=1).indices[0].tolist()

for token in top_5_tokens:
    print(f"'>>> {text.replace(tokenizer.mask_token, tokenizer.decode([token]))}'")

'>>> This is a great deal.'
'>>> This is a great success.'
'>>> This is a great adventure.'
'>>> This is a great idea.'
'>>> This is a great feat.'


In [9]:
from transformers import pipeline

fill_mask = pipeline("fill-mask", model="bert-base-uncased")

results = fill_mask("This is a great [MASK].")
print(results)

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

BertForMaskedLM LOAD REPORT from: bert-base-uncased
Key                         | Status     |  | 
----------------------------+------------+--+-
cls.seq_relationship.bias   | UNEXPECTED |  | 
bert.pooler.dense.weight    | UNEXPECTED |  | 
cls.seq_relationship.weight | UNEXPECTED |  | 
bert.pooler.dense.bias      | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

[{'score': 0.1695336550474167, 'token': 2801, 'token_str': 'idea', 'sequence': 'this is a great idea.'}, {'score': 0.06413093209266663, 'token': 2154, 'token_str': 'day', 'sequence': 'this is a great day.'}, {'score': 0.05852761119604111, 'token': 2173, 'token_str': 'place', 'sequence': 'this is a great place.'}, {'score': 0.02446654625236988, 'token': 2051, 'token_str': 'time', 'sequence': 'this is a great time.'}, {'score': 0.021729029715061188, 'token': 2518, 'token_str': 'thing', 'sequence': 'this is a great thing.'}]


README.md: 0.00B [00:00, ?B/s]

plain_text/train-00000-of-00001.parquet:   0%|          | 0.00/21.0M [00:00<?, ?B/s]

plain_text/test-00000-of-00001.parquet:   0%|          | 0.00/20.5M [00:00<?, ?B/s]

plain_text/unsupervised-00000-of-00001.p(…):   0%|          | 0.00/42.0M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating unsupervised split:   0%|          | 0/50000 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 25000
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 25000
    })
    unsupervised: Dataset({
        features: ['text', 'label'],
        num_rows: 50000
    })
})